In [1]:
import requests
import math
import pandas as pd
import os
from dotenv import load_dotenv
from datetime import datetime, timedelta
import json

load_dotenv()

KEY = os.getenv('API_KEY')
LIST_URL = "https://play.limitlesstcg.com/api/tournaments/"
# LIST_URL = "https://play.limitlesstcg.com/api/"
URL = LIST_URL+"{}"

# Tournaments List

In [2]:
start_date = datetime(2024, 12, 18).date()
end_date = datetime(2024, 12, 23).date()

In [3]:
response = requests.get(LIST_URL, headers={'X-Access-Key':KEY}, params={'game': 'Pocket', 'limit': 1000})

In [4]:
tournament_list = []
id_list = []
date_format = "%Y-%m-%dT%H:%M:%S.%fZ"
i = 0
for entry in response.json():
    # print(entry)
    date = (datetime.strptime(entry['date'], date_format) - timedelta(hours=5)).date()
    if entry['players'] >= 64 and date >= start_date and date <= end_date:
        print(entry['id'], entry['name'] )
        tournament_list.append('{}_{}'.format(i,entry['name']))
        id_list.append(entry['id'])
        i+=1


676841d086f3dba534021b50 Monday Meta Tourney #4
6763704895f729096d730ccb Christmas Special Event
674d96bd95f729096d70c355 Peladão Pocket • Stage #04
6760aee695f729096d72d080 PumaTour - TCGP ARG #2
67516a8095f729096d71235e NHKorru Community Tournament - hosted by PTCGPT
675bc03195f729096d7244b5 ♣️ Pocket Aces Turbo Weekly ♣️ 10 USD
67529b6395f729096d713fa6 Tournament Pokemon TCG Pocket Indonesia 3
6755bb4495f729096d71b4b2 Redacted Crown #12 (100 Codes) (POCKET)
6764fab995f729096d732cb2 🧪 COPA MEW 🧬| COPICLOL
675dfa1c95f729096d72928a Ursiiday's Pocket Weekly #8 • 400 USD Prize
6756108d95f729096d71ca44 #04 KGEN League BRASIL (No Ex Edition)
675d8d4595f729096d7276df The Dark League Weekend Tournament #3
673b2e71c4180e08eacfb221 $120 Pocket Champions Showdown
675746e495f729096d71e63d Pocket Legends League #12 (Season 1) - $50 USD
675ae88195f729096d722f2a 🐢Turtle Weekly 4 🐢 Mythical island legal
675f420c95f729096d72b72e 🌍 HC DUOS WORLD CUP QUALIFIER #01 🌍 $20 USD
676428c395f729096d7316e3 POC

# Tournament Data

In [4]:
for id, tour in zip(id_list, tournament_list):
    response = requests.get(URL.format(id)+"/details", headers={'X-Access-Key':KEY}).json()
    top_size = 0
    if len(response['phases']) <= 1:
        top_size = 8
    else:
        swiss_rounds = 0
        for phase in response['phases']:
            if phase['type'] != 'SWISS': break
            swiss_rounds += phase['rounds']
        

    print(response['phases'])

[{'phase': 1, 'type': 'SWISS', 'rounds': 9, 'mode': 'BO3'}, {'phase': 2, 'type': 'SINGLE_BRACKET', 'rounds': 1, 'mode': 'BO3'}]
[{'phase': 1, 'type': 'SWISS', 'rounds': 7, 'mode': 'BO3'}, {'phase': 2, 'type': 'SINGLE_ELIMINATION', 'rounds': 4, 'mode': 'BO3'}]
[{'phase': 1, 'type': 'SWISS', 'rounds': 5, 'mode': 'BO3'}, {'phase': 2, 'type': 'SINGLE_BRACKET', 'rounds': 1, 'mode': 'BO3'}]
[{'phase': 1, 'type': 'SWISS', 'rounds': 9, 'mode': 'BO3'}, {'phase': 2, 'type': 'SINGLE_BRACKET', 'rounds': 1, 'mode': 'BO3'}]
[{'phase': 1, 'type': 'SWISS', 'rounds': 8, 'mode': 'BO3'}, {'phase': 2, 'type': 'SINGLE_BRACKET', 'rounds': 1, 'mode': 'BO3'}]
[{'phase': 1, 'type': 'SWISS', 'rounds': 9, 'mode': 'BO3'}, {'phase': 2, 'type': 'SINGLE_ELIMINATION', 'rounds': 4, 'mode': 'BO3'}]
[{'phase': 1, 'type': 'SWISS', 'rounds': 7, 'mode': 'BO3'}, {'phase': 2, 'type': 'SINGLE_BRACKET', 'rounds': 1, 'mode': 'BO3'}]
[{'phase': 1, 'type': 'SWISS', 'rounds': 7, 'mode': 'BO1'}]
[{'phase': 1, 'type': 'SWISS', 'roun

In [8]:
FILENAME = "data.xlsx"
# standings = requests.get(URL.format(id_list[0])+"/standings", headers={'X-Access-Key':KEY})

deck_df = pd.DataFrame(columns=['Player', 'Nation', 'Deck', 'Tournament', 'Placement', 'Day2'])
# standing_df = pd.DataFrame(columns=['Player', 'Wins', 'Losses', 'Ties'])
# pairings_df = pd.DataFrame(columns=['Tour', 'Round', 'Player', 'Opponent', 'Result'])
matchups_df = pd.DataFrame(columns=['Deck', 'Opposing Deck', 'Wins', 'Losses', 'Ties'])

with open('archetype.json', 'r') as file:
    arch_dict = json.load(file)
    deck_df['Deck'] = deck_df['Deck'].apply(lambda x: arch_dict[x] if x in arch_dict else x)

deck_dict = deck_df.set_index('Player')['Deck'].to_dict()

ignore_list = ['67352c9357dcc5a683fc0af9', '673edab5c4180e08ead0093d']

for id, tour in zip(id_list, tournament_list):
    if id in ignore_list:
        continue
    standings = requests.get(URL.format(id)+"/standings", headers={'X-Access-Key':KEY})
    pairings = requests.get(URL.format(id)+"/pairings", headers={'X-Access-Key':KEY})
    phases = requests.get(URL.format(id)+"/details", headers={'X-Access-Key':KEY}).json()['phases']
    if not all('name' in entry['deck'] for entry in standings.json()):
        print("SKIPPED:", id, tour)
        continue

    print(tour)
    top_size = 0
    swiss_rounds = 0
    if len(phases) <= 1:
        top_size = 8
    else:
        for phase in phases:
            if phase['type'] != 'SWISS': break
            swiss_rounds += phase['rounds']
    # i = 1
    for entry in standings.json():

        # if entry['placing'] == 'None': continue
        name = "{}_{}".format(tour, entry['player'])
        nation = entry['country']
        wins = entry['record']['wins']
        losses = entry['record']['losses']
        ties = entry['record']['ties']
        # deck_df.loc[len(deck_df)] = name, entry['deck']['name']

        if top_size > 0:
            if entry['placing'] != None and entry['placing'] < (top_size+1):
                deck_df.loc[len(deck_df)] = name, nation, entry['deck']['name'], tour, "Top {}".format(pow(2, math.ceil(math.log(entry['placing'], 2)))), True
            else:
                deck_df.loc[len(deck_df)] = name, nation, entry['deck']['name'], tour, "Out of Top", False
        else:
            if entry['placing'] != None and (wins + losses + ties) > swiss_rounds:
                deck_df.loc[len(deck_df)] = name, nation, entry['deck']['name'], tour, "Top {}".format(pow(2, math.ceil(math.log(entry['placing'], 2)))), True
            else:
                deck_df.loc[len(deck_df)] = name, nation, entry['deck']['name'], tour, "Out of Top", False

    for entry in pairings.json():
        # if entry['round'] < 4:
        #     continue
        try:
            player = "{}_{}".format(tour, entry['player1'])
            opponent = "{}_{}".format(tour, entry['player2'])
            player_deck = deck_df.loc[deck_df['Player'] == player]['Deck'].values[0]
            opp_deck = deck_df.loc[deck_df['Player'] == opponent]['Deck'].values[0]
            matchup = matchups_df.loc[(matchups_df['Deck'] == player_deck) & (matchups_df['Opposing Deck'] == opp_deck)]
            inv_matchup = matchups_df.loc[(matchups_df['Deck'] == opp_deck) & (matchups_df['Opposing Deck'] == player_deck)]
            if entry['winner'] == 0:
                # pairings_df.loc[len(pairings_df)] = tour, entry['round'], player, opponent, 'T'
                # pairings_df.loc[len(pairings_df)] = tour, entry['round'], opponent, player, 'T'
                if len(matchup) == 0:
                    if player_deck != opp_deck: 
                        matchups_df.loc[len(matchups_df)] = player_deck, opp_deck, 0, 0, 1
                        matchups_df.loc[len(matchups_df)] = opp_deck, player_deck, 0, 0, 1
                    else:
                        matchups_df.loc[len(matchups_df)] = player_deck, opp_deck, 0, 0, 2
                else:
                    matchups_df.loc[matchup.index, 'Ties'] += 1
                    matchups_df.loc[inv_matchup.index, 'Ties'] += 1
            elif entry['player1'] == entry['winner']:
                # pairings_df.loc[len(pairings_df)] = tour, entry['round'], player, opponent, 'W'
                # pairings_df.loc[len(pairings_df)] = tour, entry['round'], opponent, player, 'L'
                if len(matchup) == 0:
                    if player_deck != opp_deck: 
                        matchups_df.loc[len(matchups_df)] = player_deck, opp_deck, 1, 0, 0
                        matchups_df.loc[len(matchups_df)] = opp_deck, player_deck, 0, 1, 0
                    else:
                        matchups_df.loc[len(matchups_df)] = player_deck, opp_deck, 1, 1, 0
                else:
                    matchups_df.loc[matchup.index, 'Wins'] += 1
                    matchups_df.loc[inv_matchup.index, 'Losses'] += 1
            elif entry['player2'] == entry['winner']:
                # pairings_df.loc[len(pairings_df)] = tour, entry['round'], player, opponent, 'L'
                # pairings_df.loc[len(pairings_df)] = tour, entry['round'], opponent, player, 'W'
                if len(matchup) == 0:
                    if player_deck != opp_deck: 
                        matchups_df.loc[len(matchups_df)] = player_deck, opp_deck, 0, 1, 0
                        matchups_df.loc[len(matchups_df)] = opp_deck, player_deck, 1, 0, 0
                    else:
                        matchups_df.loc[len(matchups_df)] = player_deck, opp_deck, 1, 1, 0
                else:
                    matchups_df.loc[matchup.index, 'Losses'] += 1
                    matchups_df.loc[inv_matchup.index, 'Wins'] += 1
        except:
            continue




0_#3 Cruel Monsters 🇧🇷🇧🇷
1_🌿 Team Rocket Poket Cup #14
2_TFM Pocket week 2
3_🌿 Team Rocket Poket Cup #13
4_SpX Italian Mini Cup #2
7_#3 Copa Cruel Master 🇧🇷 / Qualifier
8_Boss's Orders / 100€ Prizepool / International 🌍
9_Sky Field Skirmish Pocket Edition #2
10_SpX Italian Mini Cup #1
11_#2 Cruel Monsters 🇧🇷🇧🇷
12_Pocket Pandemonium #3
13_Monday Meta Tourney
14_Mach Punch #3
15_Peladão Pocket • Stage #01
16_#1 Cruel Monsters 🇧🇷🇧🇷
17_Pikaverse Pocket Cup #3 • 100 USD Prize
18_#01 KGEN League BRAZIL
19_Tournoi - @MrYannou (Twitch) // 50€ - FR SEULEMENT
20_Narnia's Pocket League #2
21_Ursiiday's Pocket Weekly #4 • 400 USD Prize
22_#2 Cruel Master Qualifiers 🇧🇷 / Free
23_The Wide League Friday Night Shuffle
SKIPPED: 6735bf6057dcc5a683fc18fe 24_Fu's $100 Pocket Tourney
25_Pocket Legends League #8 - 50USD (Season 1)
26_🌿 Team Rocket Poket Cup #12
27_🌿 Team Rocket Poket Cup #11
28_🌿 Team Rocket Poket Cup #10
29_Casual Pocket Gym PH #5 (WIN = INVITE)


In [ ]:
with pd.ExcelWriter(FILENAME) as writer:
    deck_df.to_excel(writer, sheet_name='decks', index=False)
    # standing_df.to_excel(writer, sheet_name='standings', index=False)
    # pairings_df.to_excel(writer, sheet_name='pairings', index=False)
    matchups_df.to_excel(writer, sheet_name='matchups', index=False)